# Intuition plot: ALM, unrolled soft loss and PAL

Trains three methods on the same 2D toy problem over several seeds and plots the constraint residuals before and after one projection step.

In [ ]:
BATCH_SIZE = 100
DEPTH = 3
WIDTH = 128
EPOCHS = 5000

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

torch.set_num_threads(1)   # reproducibility

torch.manual_seed(0)


def example_problem(x, y):
    assert x.shape == (BATCH_SIZE, 2), f"x shape: {x.shape} instead of (BATCH_SIZE, 2)"
    assert y.shape == (BATCH_SIZE, 2), f"y shape: {y.shape} instead of (BATCH_SIZE, 2)"

    # objective: per-sample squared distance, averaged over batch
    f = ((y[:, 0]-0.5) ** 2 + (y[:, 1] - 0.5) ** 2).mean()                                      

    # equality constraints: h_i(y) = 0
    h1_raw = 10 * y[:, 0]                            # linear - 1-step Newton is ~ exact 
    h2_raw = 0.01 * torch.sin(y[:, 1] * 5) + 0.1 * y[:, 0]     #  non-linear
    h1 = h1_raw.mean()
    h2 = h2_raw.mean()
    
    return f, h1, h2, h1_raw, h2_raw


def model():
    layers = []
    for i in range(DEPTH):
        in_dim = 2 if i == 0 else WIDTH
        layers += [nn.Linear(in_dim, WIDTH), nn.ReLU()]
    layers += [nn.Linear(WIDTH, 2)]
    return nn.Sequential(*layers)


def sample_conditions(seed):
    generator = torch.Generator().manual_seed(seed)
    return torch.randn(BATCH_SIZE, 2, generator=generator)

In [ ]:
t = torch.linspace(-1, 1, BATCH_SIZE)                                                                                 
                                
x = torch.stack([t, t], dim=1)                                                                                        
y = torch.stack([t, -t], dim=1)                        
                                                                                                                    
f, h1, h2, h1_raw, h2_raw = example_problem(x, y)      

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))                                                                        

ax[0].plot(t, h1_raw.detach())                                                                                        
ax[0].axhline(0, c='k', lw=.8, ls='--')                
ax[0].set(title="h1 = 10*y[:, 0]",            xlabel="y0", ylabel="h1")                                               
ax[0].grid(alpha=.3)                                                                                                  
                                                                                                                    
ax[1].plot(t, h2_raw.detach(), c='C1')                                                                                
ax[1].axhline(0, c='k', lw=.8, ls='--')                
ax[1].set(title="h2 = 0.01*sin(5*y[:, 1]) + 0.1*y[:, 0]",   xlabel="y1", ylabel="h2")                                               
ax[1].grid(alpha=.3)                                                                                                  

plt.tight_layout()                                                                                                    
plt.show()   

In [ ]:
def lm_project_step(y, constraint_stack_fn, *,                                                                        
                    delta=1e-3, lam_min=1e-6,                                                                         
                    prescale=True, prescale_floor=1.0):                                                               
    """                                                                                                               
    One K-space Levenberg-Marquardt step with Yamashita-Fukushima damping.                                            
    Equivalent to pal/projection/projector.py:_project_step_inner_lm_k for                                            
    the equality-only case (all constraints active).                                                                  
                                                                                                                
    y:                   [B, D]  live iterate (grad flows to theta).                                                  
    constraint_stack_fn: callable(y) -> [B, K]  stacked residuals.                                                    
                                                                                                                    
    Returns:                                                                                                          
    y_new:             [B, D]  detached  (one LM step from y).                                                      
    c_pre:             [B, K]  live      (= constraint_stack_fn(y)).                                                
    c_post_live:       [B, K]  live      (residuals at y_tilde_live).                                               
    disp_live:         [B, D]  live      (y - y_tilde_live).                                                        
    """                                                                                                               
    B, D = y.shape                                                                                                    
                                                                                                                    
    # Jacobian via autograd on detached y (no 2nd-order terms reach theta).
    y_det = y.detach().requires_grad_(True)                                                                           
    c_det = constraint_stack_fn(y_det)                                                                                
    K = c_det.shape[-1]                                
    J = torch.zeros(B, K, D)                                                                                          
    for k in range(K):                                                                                                
        J[:, k, :] = torch.autograd.grad(
            c_det[:, k].sum(), y_det, retain_graph=(k < K - 1),                                                       
        )[0]                                                                                                          
    J = J.detach()
                                                                                                                    
    # Live c at y (for loss + displacement path).
    c_pre = constraint_stack_fn(y)          # live                                                                    
    c_pre_det = c_pre.detach()                                                                                        
                                                                                                                    
    # Row prescale (unit-norm rows, consistent c scaling).
    B_mat, c_masked = J, c_pre_det                                                                                    
    if prescale:                                                                                                      
        row_norms = B_mat.norm(dim=2, keepdim=True).clamp(min=prescale_floor)
        B_mat    = B_mat / row_norms                                                                                  
        c_masked = c_masked / row_norms.squeeze(-1)    
                                                                                                                    
    # YF damping: lam = max(delta * ||c||^2, lam_min), per sample.
    BBT       = torch.bmm(B_mat, B_mat.transpose(1, 2))
    c_norm_sq = c_masked.pow(2).sum(dim=1, keepdim=True)                                                              
    lam_lm    = (delta * c_norm_sq).clamp(min=lam_min)                                                                
    eye       = torch.eye(K, dtype=BBT.dtype, device=BBT.device)                                                      
    BBT_reg   = BBT + lam_lm.unsqueeze(-1) * eye                                                                      
                                                                                                                    
    # Detached step (for y_new).
    L  = torch.linalg.cholesky(BBT_reg)                                                                               
    z  = torch.cholesky_solve(c_masked.unsqueeze(-1), L).squeeze(-1)                                                  
    dy = torch.bmm(B_mat.transpose(1, 2), z.unsqueeze(-1)).squeeze(-1)                                                
    y_new = y.detach() - dy                                                                                           
                                                                                                                    
    # Live displacement: same L & B_mat (detached), but live c gives
    #     gradient through y -> c -> z_live -> disp_live -> y_tilde_live.                                             
    c_live = c_pre / row_norms.squeeze(-1).clamp(min=prescale_floor) if prescale else c_pre                           
    z_live      = torch.cholesky_solve(c_live.unsqueeze(-1), L).squeeze(-1)                                           
    disp_live   = torch.bmm(B_mat.transpose(1, 2), z_live.unsqueeze(-1)).squeeze(-1)                                  
    y_tilde_live = y - disp_live                                                                                      
    c_post_live  = constraint_stack_fn(y_tilde_live)   
                                                                                                                    
    return y_new, c_pre, c_post_live, disp_live 

The three trainers: ALM with a bolt-on LM projection, an ENFORCE-style unrolled soft loss, and PAL-EMA.

In [ ]:
# MDMM dual update (Platt & Barr 1988): fixed step size, no adaptation.
# Decoupled from the quadratic penalty coefficient (RHO), which should be
# defined once outside the training loop, e.g. ETA_LAM = 1e-2, RHO = 1.0,
# and RHO replaces `mu` in the loss: penalty = lam*c + 0.5*RHO*c**2.
#
# Used instead of Basir & Senocak's APU (arXiv:2306.04904) because their variant is unstable on this specific toy problem
#
# history cols: (epoch, f, |c_pre_1|, |c_pre_2|, |c_post_1|, |c_post_2|,
#                lam0, lam1, |c_post10_1|, |c_post10_2|, f_post1, f_post10)


def train_alm(seed):
    torch.manual_seed(seed)
    net = model()
    opt = torch.optim.Adam(net.parameters(), lr=1e-4)

    K = 2
    lam = torch.zeros(K)
    # MDMM dual update hyperparameters
    ETA_LAM = 1e-2       # dual step size
    RHO     = 1.0        # fixed quadratic penalty coefficient
    # Projection hparams mirror pal_ema defaults (PALEMAConfig).
    PROJ_DELTA, PROJ_LAM_MIN = 1e-3, 1e-6

    history = []
    for epoch in range(EPOCHS):
        x = sample_conditions(seed=epoch + 100000 * seed)
        y_hat = net(x)

        # Stacked-constraint closure (x is fixed inside this loop iteration).
        cfn = lambda yy: torch.stack(example_problem(x, yy)[3:5], dim=-1)

        _, c_pre, c_post_live, _ = lm_project_step(
            y_hat, cfn, delta=PROJ_DELTA, lam_min=PROJ_LAM_MIN,
        )

        f, h1, h2, _, _ = example_problem(x, y_hat)
        penalty = (lam * torch.stack([h1, h2], dim=-1)
                   + 0.5 * RHO * torch.stack([h1, h2], dim=-1) ** 2).sum(dim=-1)
        loss = f.mean() + penalty.mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        # MDMM dual update: lambda += eta * c, no mu adaptation.
        with torch.no_grad():
            c_pre_mean = torch.stack([h1, h2]).detach()
            lam = lam + ETA_LAM * c_pre_mean

        with torch.no_grad():
            x_e = sample_conditions(seed=999)
            y_e = net(x_e)
            cfn_e = lambda yy: torch.stack(example_problem(x_e, yy)[3:5], dim=-1)
        # first eval-time projection, keep y_post1
        y_post1_e, c_pre_e, c_post_e, _ = lm_project_step(
            y_e.detach(), cfn_e, delta=PROJ_DELTA, lam_min=PROJ_LAM_MIN,
        )
        # 10-step inference-time projection
        y_iter = y_e.detach()
        for _ in range(10):
            y_iter, _, _, _ = lm_project_step(
                y_iter, cfn_e, delta=PROJ_DELTA, lam_min=PROJ_LAM_MIN,
            )
        with torch.no_grad():
            c_post10_e = cfn_e(y_iter)
            f_e, _, _, _, _ = example_problem(x_e, y_e)
            f_post1, *_ = example_problem(x_e, y_post1_e)
            f_post10, *_ = example_problem(x_e, y_iter)
            history.append((
                epoch,
                f_e.mean().item(),
                c_pre_e.abs().mean(dim=0)[0].item(),
                c_pre_e.abs().mean(dim=0)[1].item(),
                c_post_e.abs().mean(dim=0)[0].item(),
                c_post_e.abs().mean(dim=0)[1].item(),
                lam[0].item(), lam[1].item(),
                c_post10_e.abs().mean(dim=0)[0].item(),
                c_post10_e.abs().mean(dim=0)[1].item(),
                f_post1.mean().item(),
                f_post10.mean().item(),
            ))
    return np.array(history)

In [ ]:
# No multipliers, no ALM. Just:
#   loss = f(y_hat) + lambda_d * ||y_hat - y_after_10||_2
#                   + lambda_c * ||c(y_after_10)||_2
# where y_after_10 is produced by 10 chained LM projection steps with
# live gradient through the displacement chain back to y_hat.
#
# history cols: (epoch, f, |c_pre_1|, |c_pre_2|, |c_post_1|, |c_post_2|,
#                |c_post10_1|, |c_post10_2|, lam_d*disp, lam_c*cpost, f_post1, f_post10)


def train_soft(seed):
    torch.manual_seed(seed)
    net = model()
    opt = torch.optim.Adam(net.parameters(), lr=1e-4)

    PROJ_DELTA, PROJ_LAM_MIN = 1e-3, 1e-6
    PROJ_STEPS = 10
    LAMBDA_D = 0.5   # displacement L2 penalty
    LAMBDA_C = 1.0   # post-projection residual L2 penalty

    history = []
    for epoch in range(EPOCHS):
        x = sample_conditions(seed=epoch + 100000 * seed)
        y_hat = net(x)

        cfn = lambda yy: torch.stack(example_problem(x, yy)[3:5], dim=-1)

        # Chained projection with live gradient: y_hat -> y_live after PROJ_STEPS.
        y_live = y_hat
        for _ in range(PROJ_STEPS):
            _, _, _, disp_i = lm_project_step(
                y_live, cfn, delta=PROJ_DELTA, lam_min=PROJ_LAM_MIN,
            )
            y_live = y_live - disp_i

        c_post_live = cfn(y_live)           # [B, K]  live
        disp_total  = y_hat - y_live        # [B, D]  live

        f, *_ = example_problem(x, y_hat)   # f is already batch-mean scalar

        disp_l2   = (disp_total.pow(2).sum(dim=-1)  + 1e-12).sqrt()
        c_post_l2 = (c_post_live.pow(2).sum(dim=-1) + 1e-12).sqrt()

        loss = f + LAMBDA_D * disp_l2.mean() + LAMBDA_C * c_post_l2.mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        with torch.no_grad():
            x_e = sample_conditions(seed=999)
            y_e = net(x_e)
            cfn_e = lambda yy: torch.stack(example_problem(x_e, yy)[3:5], dim=-1)

        _, c_pre_e, c_post_e, _ = lm_project_step(
            y_e.detach(), cfn_e, delta=PROJ_DELTA, lam_min=PROJ_LAM_MIN,
        )
        y_iter = y_e.detach()

        # step 1: capture pre (at y_hat) and post-1
        y_iter, c_pre_e, c_post1_e, _ = lm_project_step(y_iter, cfn_e, delta=PROJ_DELTA, lam_min=PROJ_LAM_MIN)
        y_post1_e = y_iter

        # steps 2..10: just chain, don't overwrite the stored ones
        for _ in range(9):
            y_iter, _, _, _ = lm_project_step(y_iter, cfn_e, delta=PROJ_DELTA, lam_min=PROJ_LAM_MIN)

        with torch.no_grad():
            c_post10_e = cfn_e(y_iter)
            f_post1,  *_ = example_problem(x_e, y_post1_e)
            f_post10, *_ = example_problem(x_e, y_iter)
            f_e, *_   = example_problem(x_e, y_e)
            disp_l2_e   = ((y_e - y_iter).pow(2).sum(dim=-1) + 1e-12).sqrt().mean().item()
            c_post_l2_e = (c_post10_e.pow(2).sum(dim=-1)     + 1e-12).sqrt().mean().item()
            history.append((
                epoch,
                f_e.item(),
                c_pre_e.abs().mean(dim=0)[0].item(),
                c_pre_e.abs().mean(dim=0)[1].item(),
                c_post_e.abs().mean(dim=0)[0].item(),
                c_post_e.abs().mean(dim=0)[1].item(),
                c_post10_e.abs().mean(dim=0)[0].item(),
                c_post10_e.abs().mean(dim=0)[1].item(),
                LAMBDA_D * disp_l2_e,
                LAMBDA_C * c_post_l2_e,
                f_post1.mean().item(),
                f_post10.mean().item(),
            ))
    return np.array(history)

In [ ]:
# Difference from ALM in one sentence: pure `lambda * residual` loss (no mu,
# no quadratic), with a separate multiplier `lambda_d` on the displacement
# ||y_hat - y_tilde|| so the model learns to stay near the feasible manifold.
#
# The multiplier update rule is a gated EMA: after each step, look at the
# batch-mean metric. If it is above the tolerance, grow the multiplier by
# PAL_EMA_RATE times the log10 gap (capped at PAL_EMA_MAX_DECADES). The same
# rule is used for both constraint and displacement multipliers.
#
# cols: 0 epoch, 1 f, 2 f_post1, 3 f_post10, 4/5 c_pre, 6/7 c_post,
#       8/9 c_post10, 10/11 lam_c, 12 lam_d


def train_pal(seed):
    torch.manual_seed(seed)
    net = model()
    opt = torch.optim.Adam(net.parameters(), lr=1e-4)

    # Hparams (pal_ema defaults)
    proj_delta   = 1e-3     # LM damping scale:   lam_lm = delta * ||c||^2
    proj_lam_min = 1e-6     # LM damping floor
    feas_tol     = 1e-7     # |c_post| below this = "feasible"
    disp_tol     = 1e-1     # ||disp||  below this = "close enough"
    grad_clip    = 20.0

    PAL_EMA_MAX_DECADES = 1
    PAL_EMA_RATE = 1e-2
    PAL_MONOTONE = True

    # Multipliers, initialized to zero so losses ramp in organically
    lambda_c = torch.zeros(2)      # one per constraint
    lambda_d = torch.tensor(0.0)   # one for displacement magnitude

    def gated_ema_update3(lam, metric_mean, tol):
        # gap > 0 above tol (grow), gap < 0 below tol (decay), gap = 0 at tol
        gap = torch.log10(metric_mean.clamp(min=1e-30) / tol).clamp(max=PAL_EMA_MAX_DECADES)
        if PAL_MONOTONE:
            gap = gap.clamp(min=0.0)
        return (lam + PAL_EMA_RATE * gap).clamp(min=0.0)

    history = {k: [] for k in
               ["epoch", "f", "f_post1", "f_post10", "c_pre", "c_post",
                "c_post10", "lam_c", "lam_d"]}

    for epoch in range(EPOCHS):
        x = sample_conditions(seed=epoch + 100000 * seed)
        y_hat = net(x)
        cfn = lambda yy: torch.stack(example_problem(x, yy)[3:5], dim=-1)

        # One LM projection step.
        #   c_pre       : constraint value at y_hat           (live grad)
        #   c_post_live : constraint value at y_tilde         (live grad)
        #   disp_live   : y_hat - y_tilde                     (live grad)
        _, c_pre, c_post_live, disp_live = lm_project_step(
            y_hat, cfn, delta=proj_delta, lam_min=proj_lam_min,
        )

        # Per-sample magnitudes (|.| for equality; relu for inequality).
        c_pre_res  = c_pre.abs()
        c_post_res = c_post_live.abs()
        disp_mag   = disp_live.pow(2).sum(dim=-1).add(1e-12).sqrt()

        # Loss = objective + linear constraint penalty + linear displacement penalty.
        # Multipliers are detached. They don't backprop, they are updated by the rule below.
        f, *_ = example_problem(x, y_hat)
        constraint_loss   = (lambda_c.detach() * c_pre_res).sum(dim=-1).mean()
        displacement_loss = (lambda_d.detach() * disp_mag).mean()
        loss = f + constraint_loss + displacement_loss

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
        opt.step()

        # Multiplier updates driven by post-projection quantities (detached).
        with torch.no_grad():
            lambda_c = gated_ema_update3(lambda_c, c_post_res.mean(dim=0), feas_tol)
            lambda_d = gated_ema_update3(lambda_d, disp_mag.mean(),        disp_tol)

        # eval + logging
        with torch.no_grad():
            x_e = sample_conditions(seed=999)
            y_e = net(x_e)
        cfn_e = lambda yy: torch.stack(example_problem(x_e, yy)[3:5], dim=-1)

        # one LM step
        y_post1_e, c_pre_e, c_post_e, _ = lm_project_step(
            y_e.detach(), cfn_e, delta=proj_delta, lam_min=proj_lam_min,
        )
        # ten LM steps
        y_iter = y_e.detach()
        for _ in range(10):
            y_iter, _, _, _ = lm_project_step(
                y_iter, cfn_e, delta=proj_delta, lam_min=proj_lam_min,
            )
        with torch.no_grad():
            c_post10_e = cfn_e(y_iter)
            f_e, *_    = example_problem(x_e, y_e)
            f_post1,  *_ = example_problem(x_e, y_post1_e)
            f_post10, *_ = example_problem(x_e, y_iter)

            history["epoch"   ].append(epoch)
            history["f"       ].append(f_e.item())
            history["f_post1" ].append(f_post1.mean().item())
            history["f_post10"].append(f_post10.mean().item())
            history["c_pre"   ].append(c_pre_e.abs().mean(dim=0).tolist())
            history["c_post"  ].append(c_post_e.abs().mean(dim=0).tolist())
            history["c_post10"].append(c_post10_e.abs().mean(dim=0).tolist())
            history["lam_c"   ].append(lambda_c.tolist())
            history["lam_d"   ].append(lambda_d.item())

    H_pal = np.column_stack([
        history["epoch"],                   # [T]       col 0
        history["f"],                       # [T]       col 1
        history["f_post1"],                 # [T]       col 2
        history["f_post10"],                # [T]       col 3
        np.array(history["c_pre"]),         # [T, 2]    cols 4, 5
        np.array(history["c_post"]),        # [T, 2]    cols 6, 7
        np.array(history["c_post10"]),      # [T, 2]    cols 8, 9
        np.array(history["lam_c"]),         # [T, 2]    cols 10, 11
        np.array(history["lam_d"]),         # [T]       col 12
    ])
    return H_pal

Run every method over N_SEEDS seeds, aggregate in log space and plot the mean with a one-std band.

In [ ]:
import time
from tqdm.auto import tqdm

N_SEEDS = 10
TRAINERS = {"alm": train_alm, "soft": train_soft, "pal": train_pal}

H = {m: [None] * N_SEEDS for m in ("alm", "soft", "pal")}

t0 = time.time()
for method in ("alm", "soft", "pal"):
    for seed in tqdm(range(N_SEEDS), desc=method):
        H[method][seed] = TRAINERS[method](seed)
print(f"{3 * N_SEEDS} runs in {time.time() - t0:.1f}s")


In [ ]:
# Aggregate across seeds: per-seed clamp + moving average, then
# mean/std of log10 across seeds, band = 10**(mean +/- std).
from collections import deque

CLAMP = 1e-10
MA_WINDOW = EPOCHS // 40   # = 125
STRIDE = 10

# series -> (col_h1, col_h2) per method for the 4 plotted series
SERIES_COLS = {
    "alm":  {"pre": (2, 3), "post1": (4, 5)},
    "soft": {"pre": (2, 3), "post1": (4, 5)},
    "pal":  {"pre": (4, 5), "post1": (6, 7)},
}


def _moving_avg(vals, window):
    if window <= 1 or len(vals) <= window:
        return list(vals)
    out, buf = [], deque(maxlen=window)
    for v in vals:
        buf.append(v)
        out.append(sum(buf) / len(buf))
    return out


def aggregate(H):
    per_method = {}
    for method in ("alm", "soft", "pal"):
        Hs = H[method]
        epochs = Hs[0][:, 0]
        cols = SERIES_COLS[method]
        series_map = {
            "pre_h1": cols["pre"][0], "pre_h2": cols["pre"][1],
            "post_h1": cols["post1"][0], "post_h2": cols["post1"][1],
        }
        out_cols = {"epoch": epochs}
        for name, col in series_map.items():
            # per-seed clamp + moving average
            ma_stack = []
            for Hh in Hs:
                clamped = [max(v, CLAMP) for v in Hh[:, col]]
                ma = _moving_avg(clamped, MA_WINDOW)
                ma_stack.append(ma)
            ma_stack = np.array(ma_stack)            # [n_seeds, T]
            logs = np.log10(ma_stack)
            mu = logs.mean(axis=0)
            sd = logs.std(axis=0)
            out_cols[name] = 10.0 ** mu
            out_cols[name + "_lo"] = 10.0 ** (mu - sd)
            out_cols[name + "_hi"] = 10.0 ** (mu + sd)
        per_method[method] = out_cols
    return per_method


AGG = aggregate(H)

In [ ]:
# Mean line + 1 std band, 2x3 grid (rows: pre / post-1).
C_H1 = "#8B3EBC"
C_H2 = "#4FC3F7"

method_cols = [("alm", "ALM + Bolt-On"),
               ("soft", "Unrolled Soft Loss"),
               ("pal", "PAL (Ours)")]
row_defs = [("pre",  r"$h(\hat{y})$"),
            ("post", r"$h(\tilde{y}^{(1)})$")]

fig, axes = plt.subplots(2, 3, figsize=(9, 4.5), sharex=True, sharey=True)

for col, (method, title) in enumerate(method_cols):
    ag = AGG[method]
    epochs = ag["epoch"][::STRIDE]
    for row, (key, row_label) in enumerate(row_defs):
        ax = axes[row, col]
        for series, color, lbl in ((f"{key}_h1", C_H1, r"$h_1$"),
                                   (f"{key}_h2", C_H2, r"$h_2$")):
            ax.fill_between(epochs, ag[series + "_lo"][::STRIDE], ag[series + "_hi"][::STRIDE],
                            color=color, alpha=0.25, linewidth=0)
            ax.plot(epochs, ag[series][::STRIDE], color=color, linewidth=1.1, label=lbl)
        ax.set_yscale("log")
        ax.set_yticks([1e0, 1e-3, 1e-6])
        ax.set_xticks([0, 2000, 4000])
        if row == 0:
            ax.set_title(title)
        if col == 0:
            ax.set_ylabel(row_label, fontsize=11)
        if row == 1 and col == 1:
            ax.set_xlabel("Epoch")
        if row == 1 and col == 2:
            ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
plt.show()
